# Online Forum Multi-Label Communicative Type Classification

Casual online forum comments written in **Bahasa Melayu**, **English**, and **Manglish** from Lowyat Kopitiam are classified according to the communicative purpose expressed by the user.

### The 6 Communicative Categories
- **Inquiry**: Questions, seeking help, recommendations, or troubleshooting.
- **Complaint**: Venting frustration, complaining about bad service, high prices, or problems.
- **Opinion**: Personal viewpoints, beliefs, reviews, or arguments.
- **Information**: Objective facts, news, official updates, guides, or links.
- **Expressive**: Jokes, laughing, memes, greetings, or casual banter.
- **Spam**: Unwanted promotional links, referral links, or automated bot posts.

A single forum comment may serve multiple communicative purposes; therefore, **Multi-Label Classification** is used.

# 0.0 Environment Setup & Configuration
The required dependencies are installed and non-critical library warnings are suppressed for clean execution.

## 0.1 Install Required Libraries
The project dependencies are specified in `requirements.txt`.

In [ ]:
pip install -r requirements.txt

## 0.2 Suppress Warnings & Markdown Helper Setup
Configure warning filters, set Pandas column width display options to prevent truncation, and define `printmd()` using `IPython.display.Markdown`.

In [ ]:
import warnings
import pandas as pd
from IPython.display import display, Markdown, HTML

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Disable text truncation in DataFrame displays
pd.set_option('display.max_colwidth', None)

def printmd(string):
    display(Markdown(string))

# Step 1: Load Dataset & Exploratory Data Analysis (EDA)
The annotated forum dataset is loaded to inspect total records, target intent distributions, multi-label intent cardinality (how many intents each post exhibits), top intent co-occurrences, and comparative post length (word count) across all 6 communicative intent categories.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load dataset
df = pd.read_csv("dataset.csv", encoding="utf-8")

# 2. Display basic counts and top 5 rows
printmd(f"**Dataset Summary:** `{len(df):,}` total rows, `{df.shape[1]}` columns")
printmd("#### Preview of First 5 Records:")
display(df.head())

# Target intent columns
target_cols = ['Inquiry', 'Complaint', 'Opinion', 'Information', 'Expressive', 'Spam']

# 3. Individual Target Intent Distribution
intent_counts = df[target_cols].sum().sort_values(ascending=False)
printmd("#### 1. Individual Target Intent Distribution:")
display(pd.DataFrame({
    "Total Posts": intent_counts,
    "Percentage (%)": (intent_counts / len(df) * 100).round(2)
}))

# 4. Multi-Label Intent Cardinality (How many intents per post)
df['num_intents'] = df[target_cols].sum(axis=1)
intent_num_dist = df['num_intents'].value_counts().sort_index()
printmd("#### 2. Multi-Label Intent Cardinality (Number of Intents per Post):")
display(pd.DataFrame({
    "Intents per Post": intent_num_dist.index.map(lambda n: f"{n} Intent{'s' if n != 1 else ''}"),
    "Total Posts": intent_num_dist.values,
    "Percentage (%)": (intent_num_dist.values / len(df) * 100).round(2)
}))

# 5. Top Co-occurring Intent Combinations
df['intent_combination'] = df[target_cols].apply(
    lambda row: ' + '.join([col for col in target_cols if row[col] == 1]) if row.sum() > 0 else 'None (0 Intents)',
    axis=1
)
top_combinations = df['intent_combination'].value_counts().head(8)
printmd("#### 3. Top 8 Most Common Intent Combinations:")
display(pd.DataFrame({
    "Intent Combination": top_combinations.index,
    "Total Posts": top_combinations.values,
    "Percentage (%)": (top_combinations.values / len(df) * 100).round(2)
}))

# 6. Average Post Length (Word Count) per Intent Category
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))
df['char_length'] = df['text'].astype(str).apply(len)

avg_word_counts = {}
for col in target_cols:
    avg_word_counts[col] = df[df[col] == 1]['word_count'].mean()

avg_words_series = pd.Series(avg_word_counts).sort_values(ascending=False)
printmd("#### 4. Average Post Length (Word Count) per Intent Category:")
display(pd.DataFrame({"Average Word Count": avg_words_series.round(1)}))

# 7. Side-by-Side 3-Panel EDA Visualization Dashboard
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Distribution of Individual Intent Categories
sns.barplot(x=intent_counts.index, y=intent_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("1. Individual Intent Distribution", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Total Post Count")
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(intent_counts.values):
    axes[0].text(i, v + 500, f"{v:,}", ha='center', fontsize=8)

# Subplot 2: Multi-Label Intent Cardinality (Number of Intents per Post)
sns.barplot(x=[f"{n} Intent{'s' if n != 1 else ''}" for n in intent_num_dist.index], 
            y=intent_num_dist.values, ax=axes[1], palette="magma")
axes[1].set_title("2. Intents Assigned per Post", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Total Post Count")
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(intent_num_dist.values):
    axes[1].text(i, v + 500, f"{v:,}", ha='center', fontsize=8)

# Subplot 3: Average Post Length (Word Count) vs Intent Category
sns.barplot(x=avg_words_series.index, y=avg_words_series.values, ax=axes[2], palette="mako")
axes[2].set_title("3. Average Word Count vs Intent", fontsize=11, fontweight='bold')
axes[2].set_ylabel("Average Words per Post")
axes[2].tick_params(axis='x', rotation=30)
for i, v in enumerate(avg_words_series.values):
    axes[2].text(i, v + 0.5, f"{v:.1f}", ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

### Preprocessing Change Inspection Helper (Visual Colored HTML Diff View)
We define `show_step_diff()` using Python's `difflib.SequenceMatcher` and `html.escape()` to render a clean, left-aligned side-by-side HTML diff table:
- **Left-Aligned Text**: All text content is explicitly aligned to the left.
- **HTML Escaped**: Raw HTML/iframe/media tags in posts are safely escaped as plain text to prevent accidental video embeds or formatting distortions.
- **Strict Column Proportions**: Row ID (8%), BEFORE (46%), AFTER (46%).
- **Guaranteed 5-Sample Representation**: Prioritizes concise representative samples (< 250 chars), but always fills up to 5 full samples by pulling from the modified rows if fewer than 5 short posts exist.
- **Visual Highlighting**: Removed tokens in red strikethrough, normalized tokens in bold green.

In [ ]:
import difflib
import html

def show_step_diff(prev_series, curr_series, step_name):
    diff_mask = prev_series != curr_series
    total_affected = diff_mask.sum()
    printmd(f"**[{step_name}]** Rows updated: `{total_affected:,}` ({total_affected/len(prev_series)*100:.2f}%)")
    
    if total_affected == 0:
        printmd("_No rows modified in this step._")
        return
        
    changed_df = pd.DataFrame({
        "before": prev_series[diff_mask],
        "after": curr_series[diff_mask]
    })
    
    # Prioritize concise representative posts (< 250 chars), but always guarantee 5 samples if available
    short_samples = changed_df[changed_df["before"].astype(str).str.len() < 250]
    remaining_samples = changed_df[~changed_df.index.isin(short_samples.index)]
    sample_df = pd.concat([short_samples, remaining_samples]).head(5) if len(changed_df) >= 5 else changed_df
    
    html_rows = []
    for idx, row in sample_df.iterrows():
        b_words = str(row["before"]).split()
        a_words = str(row["after"]).split()
        matcher = difflib.SequenceMatcher(None, b_words, a_words)
        
        b_out, a_out = [], []
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            b_chunk = html.escape(" ".join(b_words[i1:i2]))
            a_chunk = html.escape(" ".join(a_words[j1:j2]))
            
            if tag == 'equal':
                b_out.append(b_chunk)
                a_out.append(a_chunk)
            elif tag == 'replace':
                b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
                a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
            elif tag == 'delete':
                b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
            elif tag == 'insert':
                a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
                
        html_rows.append(f'''
        <tr style="border-bottom: 1px solid #e0e0e0;">
            <td style="padding: 8px 6px; font-weight: bold; vertical-align: top; width: 8%; color: #555; text-align: left; word-break: break-word;">Row {idx}</td>
            <td style="padding: 8px 10px; vertical-align: top; width: 46%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(b_out)}</td>
            <td style="padding: 8px 10px; vertical-align: top; width: 46%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(a_out)}</td>
        </tr>
        ''')
        
    table_html = f'''
    <table style="table-layout: fixed; width: 100%; border-collapse: collapse; border: 1px solid #ccc; margin-top: 8px; margin-bottom: 16px;">
        <thead>
            <tr style="background-color: #f5f5f5; border-bottom: 2px solid #ccc; text-align: left;">
                <th style="width: 8%; padding: 8px 6px; font-size: 12px; text-align: left;">Row ID</th>
                <th style="width: 46%; padding: 8px 10px; font-size: 12px; text-align: left;">BEFORE (Prior Step)</th>
                <th style="width: 46%; padding: 8px 10px; font-size: 12px; text-align: left;">AFTER (Modified Step)</th>
            </tr>
        </thead>
        <tbody>
            {''.join(html_rows)}
        </tbody>
    </table>
    '''
    display(HTML(table_html))

# Initialize clean_text column
df['clean_text'] = df['text'].copy()

# Step 2: Remove Lowyat Quotes, BBCode, Signatures, and HTML
Forum-specific noise such as nested quote blocks, spoilers, images, BBCode tags, Lowyat edit signatures, and strict HTML markup are stripped one by one.

### 2.1 Remove Quote Blocks & Quote Headers
Quotes contain text originally written by other users, which distorts the current user's intent. Both `[quote]...[/quote]` blocks and `QUOTE(...)` headers are stripped.

In [ ]:
import re

def remove_quote_blocks(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\[quote(?:=[^\]]*)?\][\s\S]*?\[/quote\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?quote(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'QUOTE\s*\([^\)]*?\)', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.1
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_quote_blocks)
show_step_diff(prev_step, df['clean_text'], "Step 2.1: Remove Quote Blocks & Headers")

### 2.2 Remove Spoilers
Lowyat spoiler blocks `[spoiler]...[/spoiler]` and click-to-show spoiler prompts are stripped.

In [ ]:
def remove_spoilers(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'»\s*Click to show Spoiler.*?«', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[spoiler(?:=[^\]]*)?\][\s\S]*?\[/spoiler\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?spoiler(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.2
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_spoilers)
show_step_diff(prev_step, df['clean_text'], "Step 2.2: Remove Spoilers")

### 2.3 Remove Images & Code Blocks
BBCode image tags `[IMG]...[/IMG]` and code blocks `[code]...[/code]` are stripped.

In [ ]:
def remove_images_and_code(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'(?:CODE\s*)?\[IMG\][\s\S]*?\[/IMG\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?img\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[code(?:=[^\]]*)?\][\s\S]*?\[/code\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?code(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.3
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_images_and_code)
show_step_diff(prev_step, df['clean_text'], "Step 2.3: Remove Images & Code Blocks")

### 2.4 Remove Video, Attachments & Links BBCode
Strips BBCode wrappers for YouTube, embedded videos, Lowyat attachment IDs, and email tags.

In [ ]:
def remove_media_bbcode(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\[url(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/url\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?linkz(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[youtube(?:=[^\]]*)?\][\s\S]*?\[/youtube\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?youtube\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[video(?:=[^\]]*)?\][\s\S]*?\[/video\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?video\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[attachmentid=\d+\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?attachment(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Attached (?:thumbnail|image|file|picture)\(s\)', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[email(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/email\]', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_media_bbcode)
show_step_diff(prev_step, df['clean_text'], "Step 2.4: Remove Media & Attachment BBCode")

### 2.5 Remove BBCode Text Formatting Tags
Strips bold `[b]`, italic `[i]`, underline `[u]`, strikethrough `[s]`, color `[color]`, size `[size]`, and font `[font]` tags while retaining inner content words.

In [ ]:
def remove_formatting_tags(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\[/?b\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?i\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?u\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?s\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?color(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?size(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?font(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_formatting_tags)
show_step_diff(prev_step, df['clean_text'], "Step 2.5: Remove Text Formatting Tags")

### 2.6 Remove Lowyat Edit Signatures & Moderator Notices
Strips auto-generated trailing lines like `This post has been edited by...` and moderator redactions `<...removed...>`. 

In [ ]:
def remove_signatures_and_redactions(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'This post has been edited by.*?(?=\n|$)', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<[a-zA-Z0-9\s_\-\/]+removed[a-zA-Z0-9\s_\-]*>', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.6
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_signatures_and_redactions)
show_step_diff(prev_step, df['clean_text'], "Step 2.6: Remove Signatures & Moderator Notices")

### 2.7 Remove Strict HTML Tags
Removes leftover raw HTML elements (`<br>`, `<a>`, `<div>`, `<p>`, `<span>`, `<table>`, etc.).

In [ ]:
def remove_html_tags(text):
    if not isinstance(text, str): return ""
    strict_html = r'</?(?:a|abbr|b|br|button|div|em|font|h[1-6]|hr|i|iframe|img|li|ol|p|pre|s|small|span|strong|sub|sup|table|tbody|td|th|thead|tr|u|ul)(?:\s+[^>]*?)?>'
    return re.sub(strict_html, ' ', text, flags=re.IGNORECASE)

# Apply Step 2.7
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_html_tags)
show_step_diff(prev_step, df['clean_text'], "Step 2.7: Remove Strict HTML Tags")

# Step 3: Special Element Masking
Variable numerical, temporal, contact, and hyperlink expressions are mapped into normalized placeholder tokens (`URLTOKEN`, `PHONETOKEN`, `PRICETOKEN`, `DATETOKEN`, `TIMETOKEN`) to capture communicative intent without vocabulary fragmentation.

### 3.1 Mask URLs & Web Hyperlinks
Replaces HTTP/HTTPS URLs and `www.` domain links with `URLTOKEN`.

In [ ]:
def mask_urls(text):
    if not isinstance(text, str): return ""
    if not text.strip(): return text
    return re.sub(r'https?://\S+|www\.\S+', ' URLTOKEN ', text, flags=re.IGNORECASE)

# Apply Step 3.1
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_urls)
show_step_diff(prev_step, df['clean_text'], "Step 3.1: Mask URLs")

### 3.2 Mask Phone Numbers
Robustly detects and replaces Malaysian mobile and landline telephone numbers, as well as international contact numbers (using Google's `phonenumbers` library with strict prefix regex) with `PHONETOKEN`. Large standalone numbers (e.g. 140 million) and alphanumeric reference IDs are protected from false masking.

In [ ]:
from phonenumbers import PhoneNumberMatcher

def mask_phone_numbers(text, default_region="MY"):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    
    # 1. Parse via Google phonenumbers library
    try:
        matches = list(PhoneNumberMatcher(text, default_region))
        for match in reversed(matches):
            text = text[:match.start] + " PHONETOKEN " + text[match.end:]
    except Exception:
        pass
    
    # 2. Strict Malaysian mobile & landline + international regex (requires explicit phone prefix)
    text = re.sub(r'\b(?:\+?6?01)[0-46-9][-\s]?[0-9]{7,8}\b', ' PHONETOKEN ', text)
    text = re.sub(r'\b(?:\+?6?0[3-9])[-\s]?[0-9]{6,8}\b', ' PHONETOKEN ', text)
    text = re.sub(r'\+\d{1,3}[-\s]?\d{1,4}[-\s]?\d{4,8}\b', ' PHONETOKEN ', text)
    return text

# Apply Step 3.2
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_phone_numbers)
show_step_diff(prev_step, df['clean_text'], "Step 3.2: Mask Phone Numbers")

### 3.3 Mask Prices & Currency Expressions
Detects explicit currency prefixes (`RM`, `MYR`, `$`, `USD`, `SGD`, `RP`, `S$`) and currency suffixes (`sen`, `ringgit`, `dollars`) with context-aware amount masking. General numerical expressions (e.g. `5-star`, `50-10`) are preserved.

In [ ]:
CURRENCY_PREFIX = r'(?:RM|rm|MYR|myr|USD|usd|SGD|sgd|AUD|aud|RP|Rp|S\$|\$)'
CURRENCY_SUFFIX = r'(?:sen|cents?|ringgit|dollars?|myr|rm|usd|sgd|aud|rp)'

PRICE_REGEX = re.compile(
    rf'(?:'
    rf'\b{CURRENCY_PREFIX}\s*\d+(?:[\.,]\d+)?(?:\s*k\b)?|'
    rf'\b\d+(?:[\.,]\d+)?\s*{CURRENCY_SUFFIX}\b'
    rf')',
    re.IGNORECASE
)

# Explicit monetary context for numbers with 'k' (e.g. 'price 7k', 'budget 10k', 'salary 5k')
MONETARY_K_REGEX = re.compile(
    r'\b(?:price|cost|budget|salary|gaji|deposit|refund|discount)\s*(?:is|of|around|about)?\s*(\d+(?:\.\d+)?)\s*k\b',
    re.IGNORECASE
)

def mask_prices(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    text = PRICE_REGEX.sub(' PRICETOKEN ', text)
    text = MONETARY_K_REGEX.sub(' PRICETOKEN ', text)
    return text

# Apply Step 3.3
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_prices)
show_step_diff(prev_step, df['clean_text'], "Step 3.3: Mask Prices & Currencies")

### 3.4 Mask Dates
Replaces numerical calendar dates (`DD/MM/YYYY`, `DD-MM-YY`) and textual month expressions in English and Bahasa Melayu with `DATETOKEN`.

In [ ]:
MONTH_NAMES = (
    r'(?:jan(?:uary)?|januari|'
    r'feb(?:ruary)?|februari|'
    r'mar(?:ch)?|mac|'
    r'apr(?:il)?|'
    r'may|mei|'
    r'jun(?:e)?|'
    r'jul(?:y)?|julai|'
    r'aug(?:ust)?|ogos|'
    r'sep(?:t(?:ember)?)?|'
    r'oct(?:ober)?|okt(?:ober)?|'
    r'nov(?:ember)?|'
    r'dec(?:ember)?|dis(?:ember)?)'
)

DATE_REGEX = re.compile(
    rf'\b(?:0[1-9]|[12][0-9]|3[01])[-/](?:0?[1-9]|1[0-2])(?:[-/](?:\d{{2}}|\d{{4}}))?\b|'
    rf'\b[0-3]?[0-9]\s+{MONTH_NAMES}(?:\s+\d{{2,4}})?\b|'
    rf'\b{MONTH_NAMES}\s+[0-3]?[0-9](?:\s+\d{{2,4}})?\b|'
    r'\b\d{4}[-/][0-1]?[0-9][-/][0-3]?[0-9]\b',
    re.IGNORECASE
)

def mask_dates(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    return DATE_REGEX.sub(' DATETOKEN ', text)

# Apply Step 3.4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_dates)
show_step_diff(prev_step, df['clean_text'], "Step 3.4: Mask Dates")

### 3.5 Mask Timestamps
Replaces 12-hour and 24-hour time expressions (`HH:MM am/pm`, `HH:MM:SS`, `3.30pm`) with `TIMETOKEN`.

In [ ]:
TIME_REGEX = re.compile(
    r'\b(?:1[0-2]|0?[1-9]):[0-5][0-9](?::[0-5][0-9])?\s?(?:[AaPp][Mm])?\b|'
    r'\b(?:[01]?[0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9])?\b|'
    r'\b(?:1[0-2]|0?[1-9])\.[0-5][0-9](?:\.[0-5][0-9])?\s?[AaPp][Mm]\b',
    re.IGNORECASE
)

def mask_time(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    return TIME_REGEX.sub(' TIMETOKEN ', text)

# Apply Step 3.5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_time)
show_step_diff(prev_step, df['clean_text'], "Step 3.5: Mask Timestamps")

# Step 4: Convert Emojis to Text
Converts visual emojis into natural, space-separated English words using `emoji.demojize` and replaces colons and underscores with spaces (e.g. 😂 $\rightarrow$ `face with tears of joy`, 👍 $\rightarrow$ `thumbs up`). This ensures emojis become standard expressive vocabulary instead of compound tokens.

In [ ]:
from emoji import demojize

def convert_emojis(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # Convert emojis to text and replace underscores/delimiters with spaces for natural words
    converted = demojize(text, delimiters=(" ", " "))
    return converted.replace("_", " ")

# Apply Step 4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(convert_emojis)
show_step_diff(prev_step, df['clean_text'], "Step 4: Emoji Conversion (Natural Words)")

# Step 5: Lowercasing
Convert all characters to lowercase to normalize vocabulary.

In [ ]:
def to_lowercase(text):
    return text.lower() if isinstance(text, str) else ""

# Apply Step 5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(to_lowercase)
show_step_diff(prev_step, df['clean_text'], "Step 5: Lowercasing")

# Step 6: Remove Elongated Characters, Repeated Words & Repeated Phrases
Normalizes three types of repetitive noise:
1. **Elongated Characters**: Characters repeated three or more times are reduced to two occurrences (e.g. `soooo` $\rightarrow$ `soo`).
2. **Repeated Consecutive Words**: Words repeated three or more times in succession are reduced to two (e.g. `haha haha haha haha` $\rightarrow$ `haha haha`, `scam scam scam` $\rightarrow$ `scam scam`).
3. **Repeated Multi-Word Phrases**: Multi-word phrases repeated three or more times (e.g. 6 consecutive `rolling on the floor laughing` demojized emojis) are reduced to two occurrences to prevent skewed TF-IDF weights.

In [ ]:
def remove_elongated_content(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. Reduce elongated characters (>2 repetitions to 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 2. Reduce repeated consecutive words (e.g. "haha haha haha haha" -> "haha haha")
    text = re.sub(r'\b(\w+)(?:\s+\1\b){2,}', r'\1 \1', text, flags=re.IGNORECASE)
    # 3. Reduce repeated multi-word phrases (e.g. "rolling on the floor laughing" x 6 -> x 2)
    text = re.sub(r'(\b(?:\w+\s+){1,5}\w+)(?:\s+\1){2,}', r'\1 \1', text, flags=re.IGNORECASE)
    return text

# Apply Step 6
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_elongated_content)
show_step_diff(prev_step, df['clean_text'], "Step 6: Remove Elongated Characters & Repeated Phrases")

# Step 7: Build Priority-Cascading Slang Dictionary & Normalize Slangs (Hybrid Solution)

Informal Malay, Manglish, and English internet acronyms and shortforms are normalized using a multi-source dictionary cascade with an optimized **Hybrid Architecture**:
- **Multi-Word Phrases** (e.g. `apa khabar`, `hari ini`, `you all`) are normalized using a targeted regex pass.
- **Single-Word Slangs** (e.g. `sbb` $\rightarrow$ `sebab`, `xde` $\rightarrow$ `tak ada`) are normalized using an instantaneous $O(1)$ dictionary lookup, eliminating the overhead of giant 2,000-word regex patterns.

### Slang Dictionary Sources:
1. **`custom_malay_slang.json`**: Custom-curated dictionary for Malaysian forum/SMS shortforms and particle mappings (e.g. `sbb` $\rightarrow$ `sebab`, `xde` $\rightarrow$ `tiada`, `skrg` $\rightarrow$ `sekarang`, `takpe` $\rightarrow$ `tidak apa`, `lhdn`).
2. **`custom_english_slang.json`**: Custom-curated dictionary for modern forum banter and internet expressions (e.g. `topkek` $\rightarrow$ `laughing out loud`, `bye` $\rightarrow$ `goodbye`, `fyi` $\rightarrow$ `for your information`, `idk` $\rightarrow$ `i do not know`, `tbh` $\rightarrow$ `to be honest`).
3. **`malay_slangdict.json`**: Open-source Malay slang dataset from **Mendeley Data** ([Mendeley Dataset Resource](https://data.mendeley.com/datasets/mgv2n2vcb9/3/files/a7b86a2f-1175-4ff0-b813-d95218534cd4)) for broad Bahasa Melayu informal and regional spelling variations.
4. **`english_slangdict.json`**: Pre-built Internet Slang dictionary from **Ekphrasis / NoSlang** ([Ekphrasis SlangDict Source](https://github.com/cbaziotis/ekphrasis/blob/master/ekphrasis/dicts/noslang/slangdict.py)) for global English chat acronyms (`asap`, `brb`, `lol`, `rofl`).

### Priority Order Rationale:
Dictionaries are loaded in order: `custom_malay_slang.json` $\rightarrow$ `custom_english_slang.json` $\rightarrow$ `malay_slangdict.json` $\rightarrow$ `english_slangdict.json`.
If a shortform appears in multiple dictionaries, the earlier entry is preserved.

In [ ]:
import json

def build_slang_dictionary(file_paths):
    combined_dict = {}
    for path in file_paths:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                d = json.load(f)
                for k, v in d.items():
                    clean_k = str(k).strip().lower()
                    clean_v = str(v).strip().lower()
                    if clean_k and clean_k not in combined_dict:
                        combined_dict[clean_k] = clean_v
        except FileNotFoundError:
            pass
    return combined_dict

# Priority-cascaded slang dictionary list
slang_file_priority = [
    'custom_malay_slang.json',
    'custom_english_slang.json',
    'malay_slangdict.json',
    'english_slangdict.json'
]
MASTER_SLANG_DICT = build_slang_dictionary(slang_file_priority)

# Separate into multi-word phrases and single-word slangs for optimal hybrid processing
MULTI_WORD_SLANG = {k: v for k, v in MASTER_SLANG_DICT.items() if ' ' in k or '-' in k}
SINGLE_WORD_SLANG = {k: v for k, v in MASTER_SLANG_DICT.items() if ' ' not in k and '-' not in k}

printmd(f"**Total Master Slang Entries Compiled:** `{len(MASTER_SLANG_DICT):,}` entries (Single Words: `{len(SINGLE_WORD_SLANG):,}`, Phrases: `{len(MULTI_WORD_SLANG):,}`)")

# Print the complete compiled master slang dictionary
printmd("#### Full Master Slang Dictionary Mapping:")
display(pd.DataFrame(list(MASTER_SLANG_DICT.items()), columns=["Slang Key", "Normalized Replacement"]))

# Small compiled regex strictly for multi-word phrases
if MULTI_WORD_SLANG:
    phrase_keys = [re.escape(k) for k in sorted(MULTI_WORD_SLANG.keys(), key=len, reverse=True)]
    PHRASE_REGEX = re.compile(r'\b(' + '|'.join(phrase_keys) + r')\b', re.IGNORECASE)
else:
    PHRASE_REGEX = None

def normalize_slangs(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. Multi-word phrase replacement (targeted small regex)
    if PHRASE_REGEX:
        text = PHRASE_REGEX.sub(lambda m: MULTI_WORD_SLANG[m.group(0).lower()], text)
    # 2. Fast O(1) dictionary word lookup (eliminates giant regex overhead)
    words = text.split()
    return ' '.join([SINGLE_WORD_SLANG.get(w.lower(), w) for w in words])

# Apply Step 7
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(normalize_slangs)
show_step_diff(prev_step, df['clean_text'], "Step 7: Slang Normalization (Hybrid)")

# Step 8: Fix Contractions
English contractions are expanded using the `contractions` library.

In [ ]:
from contractions import fix as fix_contractions_func

def fix_contractions(text):
    return fix_contractions_func(text) if isinstance(text, str) else ""

# Apply Step 8
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(fix_contractions)
show_step_diff(prev_step, df['clean_text'], "Step 8: Fix Contractions")

# Step 9: Remove All Punctuations
All punctuation marks are stripped and replaced with spaces, ensuring zero punctuation noise survives into downstream models.

In [ ]:
import string

def remove_punctuations(text):
    if not isinstance(text, str): return ""
    punct_pattern = f"[{re.escape(string.punctuation)}]"
    return re.sub(punct_pattern, ' ', text)

# Apply Step 9
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_punctuations)
show_step_diff(prev_step, df['clean_text'], "Step 9: Remove All Punctuations")

# Step 10: Remove Non-Latin Words / Characters
Strips non-ASCII character sequences to focus corpus on Latin-scripted text.

In [ ]:
def remove_non_latin(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^\x00-\x7F]+', ' ', text)

# Apply Step 10
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_non_latin)
show_step_diff(prev_step, df['clean_text'], "Step 10: Remove Non-Latin Characters")

# Step 11: Word Tokenization
Splits cleaned text into individual word tokens using NLTK `word_tokenize`.

In [ ]:
from nltk import download as nltk_download
from nltk.tokenize import word_tokenize

# Download NLTK tokenization models
nltk_download('punkt', quiet=True)
nltk_download('punkt_tab', quiet=True)
nltk_download('words', quiet=True)

def tokenize_words(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return word_tokenize(text)

# Apply Step 11
df['tokens'] = df['clean_text'].apply(tokenize_words)

printmd("**Step 11 Completed:** Tokenized cleaned text into token lists.")
sample_tokens_df = pd.DataFrame({
    "Raw Clean Text": df['clean_text'].head(5),
    "Tokens Output": df['tokens'].head(5)
})
display(sample_tokens_df)

# Step 12: Language Classification and Token Tagging (`malaya.dictionary`)
Tokens are classified using **Malaya's dictionary identifiers** (`from malaya.dictionary import is_malay, is_english`), tagging tokens as `TAG`, `MALAY`, `ENGLISH`, or `UNKNOWN` to guide downstream language-specific morphological processing.

In [ ]:
from malaya.dictionary import is_malay, is_english

MASKING_TAGS = {'urltoken', 'phonetoken', 'pricetoken', 'timetoken', 'datetoken'}

def tag_tokens(tokens):
    tagged = []
    for token in tokens:
        token_lower = token.lower()
        if token_lower in MASKING_TAGS:
            tagged.append((token, "TAG"))
        elif is_malay(token_lower):
            tagged.append((token, "MALAY"))
        elif is_english(token_lower):
            tagged.append((token, "ENGLISH"))
        else:
            tagged.append((token, "UNKNOWN"))
    return tagged

# Apply Step 12
df['tagged_tokens'] = df['tokens'].apply(tag_tokens)

printmd("**Step 12 Completed:** Tagged language and entity categories on tokens via `malaya.dictionary`.")
sample_tagged_df = pd.DataFrame({
    "Tokens": df['tokens'].head(5),
    "Tagged Tokens": df['tagged_tokens'].head(5)
})
display(sample_tagged_df)

# Step 13: Stop Word Removal
Removes English stopwords (via **SpaCy**) and Malay stopwords (via **Malaya**), while stripping isolated single-character noise words (e.g. isolated `'i'`). All active English and Malay stopwords are printed below for inspection.

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOPWORDS
from malaya.text.function import stopwords

# Malaya stopwords is already a list object
MALAYA_STOPWORDS = set(stopwords)

# Display all Stopword sets
printmd(f"#### Active Stopword Catalogs:")
printmd(f"**English Stopwords ({len(SPACY_STOPWORDS)} words via SpaCy):**\n`{', '.join(sorted(SPACY_STOPWORDS))}`")
printmd(f"**Malay Stopwords ({len(MALAYA_STOPWORDS)} words via Malaya):**\n`{', '.join(sorted(MALAYA_STOPWORDS))}`")

def remove_stopwords(tagged_tokens):
    cleaned = []
    for word, tag in tagged_tokens:
        word_lower = word.lower()
        if tag == "ENGLISH" and word_lower in SPACY_STOPWORDS:
            continue
        elif tag == "MALAY" and word_lower in MALAYA_STOPWORDS:
            continue
        # Also drop single-letter noise words (e.g. isolated 'i')
        if len(word) > 1 or word.isdigit() or word_lower in MASKING_TAGS:
            cleaned.append((word, tag))
    return cleaned

# Apply Step 13
df['filtered_tokens'] = df['tagged_tokens'].apply(remove_stopwords)
printmd("**Step 13 Completed:** Filtered out stopwords and single-character noise.")
sample_stopwords_df = pd.DataFrame({
    "Before (Tagged)": df['tagged_tokens'].head(5),
    "After (Filtered)": df['filtered_tokens'].head(5)
})
display(sample_stopwords_df)

# Step 14: Morphological Normalization (Lemmatization & Stemming)
English tokens are lemmatized using **WordNetLemmatizer**; Malay tokens are stemmed using **Sastrawi**. The resulting token list is flattened into the final `clean_text` corpus.

In [ ]:
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk_download('wordnet', quiet=True)
lemmatizer = WordNetLemmatizer()
stemmer_factory = StemmerFactory()
malay_stemmer = stemmer_factory.create_stemmer()

def lemmatize_and_stem(tagged_tokens):
    processed = []
    for word, tag in tagged_tokens:
        if tag == "ENGLISH":
            processed.append((lemmatizer.lemmatize(word), tag))
        elif tag == "MALAY":
            processed.append((malay_stemmer.stem(word), tag))
        else:
            processed.append((word, tag))
    return processed

# Apply Step 14
df['morph_tokens'] = df['filtered_tokens'].apply(lemmatize_and_stem)
df['clean_text'] = df['morph_tokens'].apply(lambda t_list: ' '.join([t[0] for t in t_list]))
printmd("**Step 14 Completed:** Normalized morphology (lemmatization and stemming).")
sample_morph_df = pd.DataFrame({
    "Original Post": df['text'].head(5),
    "Final Clean Text": df['clean_text'].head(5)
})
display(sample_morph_df)

# Step 14.1: Filter Out Empty & 0-Token Posts
Posts that contained exclusively quotes, spoilers, stripped media, or stopwords are reduced to 0 tokens / empty strings. Filtering them out prevents zero-vector noise from degrading TF-IDF feature weights and classifier decision boundaries.

In [ ]:
initial_total_rows = len(df)

# Filter for posts containing at least 1 valid cleaned token
df = df[df['clean_text'].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
dropped_rows = initial_total_rows - len(df)

printmd(f"**Step 14.1 Completed:** Dropped `{dropped_rows:,}` empty/0-token posts ({dropped_rows/initial_total_rows*100:.2f}% of total).")
printmd(f"**Cleaned Dataset for Model Training:** `{len(df):,}` valid informative posts remaining.")

# Step 15: Train / Test Dataset Splitting
The preprocessed corpus is partitioned into a **70% training set** and a **30% testing set** using stratified multi-label sampling.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df[target_cols], test_size=0.30, random_state=42
)

printmd(f"**Dataset Partition:** Training set: `{len(X_train):,}` samples ({len(X_train)/len(df)*100:.1f}%), Testing set: `{len(X_test):,}` samples ({len(X_test)/len(df)*100:.1f}%)")
printmd("#### Class Distribution Across Partitions:")
split_summary_df = pd.DataFrame({
    "Training Set Count": y_train.sum(),
    "Testing Set Count": y_test.sum(),
    "Total Count": df[target_cols].sum()
})
display(split_summary_df)

# Step 16: TF-IDF Vectorization
We fit `TfidfVectorizer` (unigrams + bigrams, `min_df=3`) on the training set, transform the test set, and persist `tfidf_vectorizer.pkl`. Scikit-learn's internal stopword removal is disabled (`stop_words=None`) since domain-specific bilingual stopwords were already filtered in Step 13.

In [ ]:
from joblib import dump as joblib_dump
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words=None, min_df=3, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Persist TF-IDF vectorizer artifact
joblib_dump(tfidf, "tfidf_vectorizer.pkl")

printmd(f"**TF-IDF Vocabulary Space:** Fitted `{len(tfidf.get_feature_names_out()):,}` unique n-gram features across `{X_train_tfidf.shape[0]:,}` training documents.")

# Step 17: Train One-vs-Rest Logistic Regression Model
Train One-vs-Rest Logistic Regression with balanced class weights to handle class imbalance across all 6 communicative intent categories.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

lr_model = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
lr_model.fit(X_train_tfidf, y_train)

# Persist Logistic Regression model artifact
joblib_dump(lr_model, "logistic_regression_model.pkl")
printmd("**Logistic Regression Model:** Successfully trained and saved to `logistic_regression_model.pkl`.")

# Step 18: Logistic Regression Feature Interpretation
Inspect the highest positive coefficient weights ($w_j$) for each communicative intent category to verify that the model relies on meaningful linguistic markers.

In [ ]:
feature_names = tfidf.get_feature_names_out()
top_words_lr = {}
for i, label in enumerate(target_cols):
    coefs = lr_model.estimators_[i].coef_[0]
    top_indices = np.argsort(coefs)[-8:][::-1]
    top_words_lr[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in top_indices]
printmd("#### Top 8 Predictive Features per Intent Class (Logistic Regression):")
display(pd.DataFrame(top_words_lr))


# Step 19: Logistic Regression Decision Boundary Equations & Evaluation
We extract the linear decision boundary equations and evaluate performance using Confusion Matrix heatmaps and Classification Reports.

### Linear Score & Sigmoid Probability Formulas:
$$z_k = w_{0,k} + \sum_{j=1}^{d} w_{j,k} x_j$$
$$P(Y_k = 1 \mid \mathbf{x}) = \frac{1}{1 + e^{-z_k}}$$

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

printmd("#### Logistic Regression Linear Decision Boundary Equations:")
for i, label in enumerate(target_cols):
    coefs = lr_model.estimators_[i].coef_[0]
    intercept = lr_model.estimators_[i].intercept_[0]
    top_3_idx = np.argsort(coefs)[-3:][::-1]
    terms = " + ".join([f"({coefs[idx]:.2f} * {feature_names[idx]})" for idx in top_3_idx])
    print(f"[{label:<11}] z = {intercept:.2f} + {terms} + ...")
    
y_pred_lr = lr_model.predict(X_test_tfidf)

# Plot Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
lr_f1_scores, lr_accuracies, lr_precisions, lr_recalls = [], [], [], []

for i, label in enumerate(target_cols):
    cm = confusion_matrix(y_test[label], y_pred_lr[:, i])
    tn, fp = cm[0]
    fn, tp = cm[1]
    
    acc = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    lr_accuracies.append(acc)
    lr_precisions.append(prec)
    lr_recalls.append(rec)
    lr_f1_scores.append(f1)
    
    sns.heatmap(cm, ax=axes[i], annot=True, fmt="d", cmap="Blues",
                xticklabels=[f"Pred No {label}", f"Pred {label}"],
                yticklabels=[f"Act No {label}", f"Act {label}"])
    axes[i].set_title(f"Logistic Regression: {label} (F1: {f1:.3f})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

printmd("#### Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=target_cols))

# Step 20: Train One-vs-Rest Random Forest Model
Train One-vs-Rest Random Forest (100 decision trees, `max_depth=20`) as a non-linear ensemble baseline.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = OneVsRestClassifier(RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1))
rf_model.fit(X_train_tfidf, y_train)

# Persist Random Forest model artifact
joblib_dump(rf_model, "random_forest_model.pkl")
printmd("**Random Forest Model:** Successfully trained and saved to `random_forest_model.pkl`.")

# Step 21: Random Forest Feature Importances & Evaluation
Inspect top Gini feature importances and evaluate test set predictions with Confusion Matrices and Classification Reports.

In [ ]:
top_words_rf = {}
for i, label in enumerate(target_cols):
    importances = rf_model.estimators_[i].feature_importances_
    top_indices = np.argsort(importances)[-8:][::-1]
    top_words_rf[label] = [f"{feature_names[idx]} ({importances[idx]:.4f})" for idx in top_indices]
printmd("#### Random Forest Top Gini Feature Importances:")
display(pd.DataFrame(top_words_rf))

y_pred_rf = rf_model.predict(X_test_tfidf)

# Plot Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
rf_f1_scores, rf_accuracies, rf_precisions, rf_recalls = [], [], [], []

for i, label in enumerate(target_cols):
    cm = confusion_matrix(y_test[label], y_pred_rf[:, i])
    tn, fp = cm[0]
    fn, tp = cm[1]
    
    acc = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    rf_accuracies.append(acc)
    rf_precisions.append(prec)
    rf_recalls.append(rec)
    rf_f1_scores.append(f1)
    
    sns.heatmap(cm, ax=axes[i], annot=True, fmt="d", cmap="Greens",
                xticklabels=[f"Pred No {label}", f"Pred {label}"],
                yticklabels=[f"Act No {label}", f"Act {label}"])
    axes[i].set_title(f"Random Forest: {label} (F1: {f1:.3f})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

printmd("#### Classification Report for Random Forest:")
print(classification_report(y_test, y_pred_rf, target_names=target_cols))

# Step 22: Comprehensive All-Metrics Model Comparison
Consolidated comparison of **Accuracy**, **Precision**, **Recall**, and **F1-Score** between Logistic Regression and Random Forest across all 6 intent categories + Macro Average.

In [ ]:
# Consolidated All-Metrics Comparison Table
comp_table = pd.DataFrame({
    'Category': target_cols,
    'LR Accuracy': lr_accuracies,
    'RF Accuracy': rf_accuracies,
    'LR Precision': lr_precisions,
    'RF Precision': rf_precisions,
    'LR Recall': lr_recalls,
    'RF Recall': rf_recalls,
    'LR F1-Score': lr_f1_scores,
    'RF F1-Score': rf_f1_scores,
})

# Add Macro Average Row
comp_table.loc[len(comp_table)] = {
    'Category': 'MACRO AVERAGE',
    'LR Accuracy': np.mean(lr_accuracies),
    'RF Accuracy': np.mean(rf_accuracies),
    'LR Precision': np.mean(lr_precisions),
    'RF Precision': np.mean(rf_precisions),
    'LR Recall': np.mean(lr_recalls),
    'RF Recall': np.mean(rf_recalls),
    'LR F1-Score': np.mean(lr_f1_scores),
    'RF F1-Score': np.mean(rf_f1_scores),
}

printmd("### Overall Model Performance Comparison (Logistic Regression vs Random Forest)")
display(comp_table.style.format({col: "{:.3f}" for col in comp_table.columns if col != 'Category'}))

# Side-by-Side Multi-Metric Visual Comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics_to_plot = [
    ("Accuracy", "LR Accuracy", "RF Accuracy"),
    ("Precision", "LR Precision", "RF Precision"),
    ("Recall", "LR Recall", "RF Recall"),
    ("F1-Score", "LR F1-Score", "RF F1-Score")
]

x = np.arange(len(target_cols))
width = 0.35

for idx, (m_name, lr_c, rf_c) in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    ax.bar(x - width/2, comp_table.iloc[:6][lr_c], width, label="Logistic Regression", color="#4C72B0")
    ax.bar(x + width/2, comp_table.iloc[:6][rf_c], width, label="Random Forest", color="#55A868")
    ax.set_title(f"Model Comparison: {m_name} per Category", fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(target_cols, rotation=25)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Step 23: Preprocessing Pipeline Encapsulation & Model Artifact Loading
We encapsulate Steps 2 through 14 into a clean standalone `preprocess_pipeline(raw_text)` function and load the saved `.pkl` model artifacts from disk for deployment.

In [ ]:
from joblib import load as joblib_load

# Encapsulate Steps 2 to 14
def preprocess_pipeline(raw_text):
    # Step 2: Structural cleaning
    text = remove_quote_blocks(raw_text)
    text = remove_spoilers(text)
    text = remove_images_and_code(text)
    text = remove_media_bbcode(text)
    text = remove_formatting_tags(text)
    text = remove_signatures_and_redactions(text)
    text = remove_html_tags(text)
    
    # Step 3: Special element masking
    text = mask_urls(text)
    text = mask_phone_numbers(text)
    text = mask_prices(text)
    text = mask_dates(text)
    text = mask_time(text)
    
    # Steps 4 to 10: Normalization
    text = convert_emojis(text)
    text = to_lowercase(text)
    text = remove_elongated_content(text)
    text = normalize_slangs(text)
    text = fix_contractions(text)
    text = remove_punctuations(text)
    text = remove_non_latin(text)
    
    # Steps 11 to 14: Morphology & Stopwords
    tokens = tokenize_words(text)
    tagged = tag_tokens(tokens)
    filtered = remove_stopwords(tagged)
    morph_tokens = lemmatize_and_stem(filtered)
    
    return ' '.join([t[0] for t in morph_tokens])

# Load serialized artifacts from disk
loaded_tfidf = joblib_load("tfidf_vectorizer.pkl")
loaded_lr = joblib_load("logistic_regression_model.pkl")
loaded_rf = joblib_load("random_forest_model.pkl")

printmd("**Artifacts Status:** All serialized models (`TF-IDF`, `Logistic Regression`, `Random Forest`) loaded successfully from disk.")

# Step 24: Real-Time Prediction on Unseen Posts (With Interactive Input While-Loop)
Execute real-time multi-label classification on brand-new unseen forum posts. You can run automated sample tests or enter comments interactively (which loops continuously until typing `'q'` to exit).

In [ ]:
def predict_unseen(raw_text, model_type="logistic_regression"):
    cleaned_input = preprocess_pipeline(raw_text)
    
    if not cleaned_input.strip():
        return {
            "raw_text": raw_text,
            "cleaned_text": cleaned_input,
            "predicted_intents": ["Neutral/None"]
        }
    
    vectorized_input = loaded_tfidf.transform([cleaned_input])
    
    if model_type == "logistic_regression":
        pred_array = loaded_lr.predict(vectorized_input)[0]
    elif model_type == "random_forest":
        pred_array = loaded_rf.predict(vectorized_input)[0]
    else:
        raise ValueError("Invalid model_type. Choose 'logistic_regression' or 'random_forest'.")
    
    predicted_intents = [target_cols[i] for i, val in enumerate(pred_array) if val == 1]
    
    return {
        "raw_text": raw_text,
        "cleaned_text": cleaned_input,
        "predicted_intents": predicted_intents if predicted_intents else ["Neutral/None"]
    }

# 1. Automated Test Verification on Sample Forum Comments
test_samples = [
    "QUOTE(seller @ 10am) barang rosak teruk refund RM50 https://shop.com/scam please help!!!",
    "bila tarikh release movie baru tu? nak book ticket kat shopee",
    "i",
    "may i know which course you chosing"
]

printmd("#### Automated Verification on Test Sample Posts:")
for post in test_samples:
    result = predict_unseen(post, model_type="logistic_regression")
    print(f"Post  : {result['raw_text']}")
    print(f"Clean : {result['cleaned_text']}")
    print(f"Intent: {result['predicted_intents']}\n")

# 2. Interactive Input While-Loop (Exits only on 'q')
def interactive_intent_predictor():
    printmd("### Interactive Forum Comment Intent Predictor")
    printmd("Type any forum comment below to test intent classification (or enter '**q**' to quit):")
    
    while True:
        user_post = input("\nYour comment (or 'q' to quit): ").strip()
        if user_post.lower() == 'q':
            printmd("**Exiting interactive session.**")
            break
        if not user_post:
            print("Please enter a non-empty comment.")
            continue
            
        model_choice = input("Select model (lr / rf, default: lr): ").strip().lower()
        m_type = "random_forest" if model_choice == "rf" else "logistic_regression"
        
        res = predict_unseen(user_post, model_type=m_type)
        printmd("#### Prediction Output:")
        printmd(f"- **Model Used:** {'Logistic Regression' if m_type == 'logistic_regression' else 'Random Forest'}")
        printmd(f"- **Raw Input:** `{res['raw_text']}`")
        printmd(f"- **Cleaned Input:** `{res['cleaned_text']}`")
        printmd(f"- **Predicted Intents:** **{', '.join(res['predicted_intents'])}**")

# Run interactive predictor
interactive_intent_predictor()